In [0]:
# Bronze (External Volume)
bronze_aw2_path = "/Volumes/main/default/fabmigration1_bronze/adventureworks/"

df_aw2_bronze = spark.read.parquet(bronze_aw2_path)

In [0]:
def clean_df(df):
    # trim() a strings
    for col_name, dtype in df.dtypes:
        if dtype == "string":
            df = df.withColumn(col_name, F.trim(F.col(col_name)))

    # Replace nulls
    fill_dict = {col: "" for col, dtype in df.dtypes if dtype == "string"}
    fill_dict_numeric = {col: 0 for col, dtype in df.dtypes if dtype in ["int", "bigint", "double"]}
    df = df.fillna(fill_dict).fillna(fill_dict_numeric)

    return df

Parquet files without schema

In [0]:
from typing import List, Optional
from pyspark.sql import DataFrame, functions as F, types as T
from pyspark.databricks.sql import functions as dbf
import re

def standardizedate_df(
    df: DataFrame,
    date_cols: Optional[List[str]] = None,
    input_formats: Optional[List[str]] = None,
    cast_dates_as: str = "date",     # "date" or "timestamp"
    ingest_date: Optional[str] = None,  # e.g., "2026-02-03" (casts to date); if None -> current_date()
    keep_raw: bool = True,              # guarda valor original
    add_parse_flag: bool = True          # flag de parse fallido
) -> DataFrame:
    """
    1) Standardizes the provided or auto-detected date/timestamp columns:
       - Parses common string formats to timestamp
       - Casts to `date` (default) or keeps as `timestamp` via `cast_dates_as`
    2) Adds `ingest_date` column (date) for downstream partitioning.

    Parameters
    ----------
    df : input DataFrame
    date_cols : list of column names to treat as dates/timestamps. If None, auto-detects by name/ctype.
    input_formats : list of strptime-style formats to try in order
    cast_dates_as : "date" or "timestamp"
    ingest_date : string literal yyyy-MM-dd; if None uses current_date()
    """

    # --- 0) Trim strings
    for col_name, dtype in df.dtypes:
        if dtype == "string":
            df = df.withColumn(col_name, F.trim(F.col(col_name)))

    # --- 1) Which columns are date-like?
    if date_cols is None:
        # Heuristic: names that look like dates & are string/date/timestamp
        pattern = re.compile(r"(^|_)(date|dt|ts|timestamp|fecha)(_|\b)")
        date_cols = [
            c for c, t in df.dtypes
            if t in ("string", "date", "timestamp")
            and pattern.search(c.lower())
        ]

    # --- 2) Parse formats for string dates
    if input_formats is None:
        input_formats = [
            # Date
            "yyyy-MM-dd", "yyyy/MM/dd",
            "dd-MM-yyyy", "dd/MM/yyyy",
            "MM-dd-yyyy", "MM/dd/yyyy",
            "yyyyMMdd", "ddMMyyyy",

            # Date + time
            "yyyy-MM-dd HH:mm:ss",
            "dd/MM/yyyy HH:mm:ss",
            "MM/dd/yyyy HH:mm:ss",

            # ISO 8601
            "yyyy-MM-dd'T'HH:mm:ss",
            "yyyy-MM-dd'T'HH:mm:ss.SSS",
            "yyyy-MM-dd'T'HH:mm:ssX",
            "yyyy-MM-dd'T'HH:mm:ssXX",
            "yyyy-MM-dd'T'HH:mm:ssXXX",
            "yyyy-MM-dd'T'HH:mm:ss.SSSX",
            "yyyy-MM-dd'T'HH:mm:ss.SSSXX",
            "yyyy-MM-dd'T'HH:mm:ss.SSSXXX"
        ]

    dtype_map = dict(df.dtypes)


    # ---------------------------------------------------
    # 3) Parseo seguro por columna
    # ---------------------------------------------------
    for c in date_cols:
        t = dtype_map.get(c, "")

        if t == "string":

            # --- Preservar valor original
            if keep_raw:
                df = df.withColumn(f"{c}__raw", F.col(c))

            # --- Parse por formatos (ANSI-safe)
            parsed = None
            for fmt in input_formats:
                candidate = dbf.try_to_timestamp(F.col(c), F.lit(fmt))
                parsed = candidate if parsed is None else F.coalesce(parsed, candidate)

            # --- Epoch seconds / millis (solo si es numérico)
            digits = F.regexp_replace(F.col(c), r"\D", "")
            epoch_ts = (
                F.when(F.length(digits) == 13,
                       F.to_timestamp(F.from_unixtime(digits.cast("double") / 1000)))
                 .when(F.length(digits) == 10,
                       F.to_timestamp(F.from_unixtime(digits.cast("double"))))
            )

            parsed = F.coalesce(parsed, epoch_ts)

            # --- Asignar: si no parsea → NULL
            df = df.withColumn(c, parsed)

            # --- Flag de error de parseo
            if add_parse_flag:
                df = df.withColumn(
                    f"{c}__parse_failed",
                    F.col(c).isNull() & F.col(f"{c}__raw").isNotNull()
                )

        # ---------------------------------------------------
        # 4) Normalización final del tipo
        # ---------------------------------------------------
        if cast_dates_as == "date":
            df = df.withColumn(c, F.to_date(F.col(c)))
        else:
            df = df.withColumn(c, F.col(c).cast("timestamp"))

    # ---------------------------------------------------
    # 5) ingest_date (para particionado)
    # ---------------------------------------------------
    if ingest_date:
        df = df.withColumn("ingest_date", F.lit(ingest_date).cast(T.DateType()))
    else:
        df = df.withColumn("ingest_date", F.current_date())

    return df


In [0]:
from pyspark.sql.functions import col, lit
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, BooleanType, DecimalType, TimestampType

# -------------------------------------------------------------
# 0) Paths 
# -------------------------------------------------------------
account   = "synpasetofabric"
container = "fabmigation1"
schema_name = "Production"
table_name  = "Product"

#bronze_path = f"abfss://{container}@{account}.dfs.core.windows.net/bronze/adventureworks/{schema_name}.{table_name}.parquet/"
#base_silver_root = f"abfss://{container}@{account}.dfs.core.windows.net/silver/adventureworks/"

bronze_path = f"/Volumes/main/default/fabmigration1_bronze/adventureworks/{schema_name}.{table_name}.parquet/"
base_silver_root = f"main.silver.{schema_name}_{table_name}" 


print("BRONZE:", bronze_path)
print("SILVER:", base_silver_root)

silver_path   = base_silver_root
display(silver_path)

In [0]:
from pyspark.sql.functions import col, lit
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, BooleanType, DecimalType, TimestampType

# -------------------------------------------------------------
# 0) Paths 
# -------------------------------------------------------------
account   = "synpasetofabric"
container = "fabmigation1"
schema_name = "Production"
table_name  = "Product"

bronze_path = f"/Volumes/main/default/fabmigration1_bronze/adventureworks/{schema_name}.{table_name}.parquet/"
base_silver_root = f"main.silver.{schema_name}_{table_name}" 

#print("BRONZE:", bronze_path)
#print("SILVER:", base_silver_root)

silver_path   = base_silver_root
#display(silver_path)

# -------------------------------------------------------------
# 1) Tu SCHEMA (ya lo tienes definido como schemaProduct)
#    *Asegúrate de que esté en el notebook antes de este bloque*
# -------------------------------------------------------------
schemaProduct = StructType([
    StructField("ProductID",               IntegerType(),    True),
    StructField("Name",                    StringType(),     True),
    StructField("ProductNumber",           StringType(),     True),
    StructField("MakeFlag",                BooleanType(),    True),
    StructField("FinishedGoodsFlag",       BooleanType(),    True),
    StructField("Color",                   StringType(),     True),
    StructField("SafetyStockLevel",        IntegerType(),    True),
    StructField("ReorderPoint",            IntegerType(),    True),
    StructField("StandardCost",            DecimalType(19,4),True),
    StructField("ListPrice",               DecimalType(19,4),True),
    StructField("Size",                    StringType(),     True),
    StructField("SizeUnitMeasureCode",     StringType(),     True),
    StructField("WeightUnitMeasureCode",   StringType(),     True),
    StructField("Weight",                  DecimalType(8,2), True),
    StructField("DaysToManufacture",       IntegerType(),    True),
    StructField("ProductLine",             StringType(),     True),
    StructField("Class",                   StringType(),     True),
    StructField("Style",                   StringType(),     True),
    StructField("ProductSubcategoryID",    IntegerType(),    True),
    StructField("ProductModelID",          IntegerType(),    True),
    StructField("SellStartDate",           TimestampType(),  True),
    StructField("SellEndDate",             TimestampType(),  True),
    StructField("DiscontinuedDate",        TimestampType(),  True),
    StructField("rowguid",                 StringType(),     True),
    StructField("ModifiedDate",            TimestampType(),  True)
])

# Final column order according to your schema
final_cols = [f.name for f in schemaProduct]

# -------------------------------------------------------------
# 2) Read parquet
# -------------------------------------------------------------
df_bronze = spark.read.parquet(bronze_path)

# -------------------------------------------------------------
# 3) Align column names
#    - If the Parquet already has the same number of columns but with
#      different/abbreviated names, rename them following schemaProduct.
#    - If columns are missing, create them as null (cast to the correct type).
# -------------------------------------------------------------
df = df_bronze

existing_cols = df.columns

# 3a) If the number of columns matches, rename by position (safe when coming from ORC/Parquet without "correct" headers)
if len(existing_cols) == len(final_cols):
    for old_name, new_name in zip(existing_cols, final_cols):
        if old_name != new_name:
            df = df.withColumnRenamed(old_name, new_name)

# 3b) Create columns as null
for f in final_cols:
    if f not in df.columns:
        df = df.withColumn(f, lit(None).cast([sf.dataType for sf in schemaProduct if sf.name == f][0]))

# -------------------------------------------------------------
# 4) Cast types as schemaProduct
# -------------------------------------------------------------
for sf in schemaProduct:
    if sf.name in df.columns:
        df = df.withColumn(sf.name, col(sf.name).cast(sf.dataType))

# -------------------------------------------------------------
# 5) Reorder columns and write to silver in Parquet
#    - Snappy by default; you can change the compression if you want
# -------------------------------------------------------------
df_final = df.select([col(c) for c in final_cols])

#clean
df_clean = clean_df(df_final)
# dates
df_table= standardizedate_df(df_clean)
#display(df_table)

# Save  (Silver)  
df_table.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_path)             

In [0]:
from pyspark.sql.functions import col, lit
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType

# -------------------------------------------------------------
# 0) Paths
# -------------------------------------------------------------
account   = "synpasetofabric"
container = "fabmigation1"
schema_name = "Production"
table_name  = "ProductModelProductDescriptionCulture"

bronze_path = f"/Volumes/main/default/fabmigration1_bronze/adventureworks/{schema_name}.{table_name}.parquet/"
base_silver_root = f"main.silver.{schema_name}_{table_name}" 

print("BRONZE:", bronze_path)
print("SILVER:", base_silver_root)

silver_path   = base_silver_root
display(silver_path)


schemaPMDC = StructType([
    StructField("ProductModelID",        IntegerType(),   True),
    StructField("ProductDescriptionID",  IntegerType(),   True),
    StructField("CultureID",             StringType(),    True),   # nchar(6) → String
    StructField("ModifiedDate",          TimestampType(), True)
])

final_cols = [f.name for f in schemaPMDC]

df_bronze = spark.read.parquet(bronze_path)
# -------------------------------------------------------------
# 3) Align column names
#    - If the Parquet already has the same number of columns but with
#      different/abbreviated names, rename them following schemaProduct.
#    - If columns are missing, create them as null (cast to the correct type).
# -------------------------------------------------------------
df = df_bronze
existing_cols = df.columns

# 4a) Si coinciden en cantidad, renombrar por posición
if len(existing_cols) == len(final_cols):
    for old, new in zip(existing_cols, final_cols):
        if old != new:
            df = df.withColumnRenamed(old, new)

# 4b) Create columns as null
for f in final_cols:
    if f not in df.columns:
        dtype = [sf.dataType for sf in schemaPMDC if sf.name == f][0]
        df = df.withColumn(f, lit(None).cast(dtype))

for sf in schemaPMDC:
    if sf.name in df.columns:
        df = df.withColumn(sf.name, col(sf.name).cast(sf.dataType))

df_final = df.select([col(c) for c in final_cols])

# Clean (tu funciones)
df_clean = clean_df(df_final)
df_table = standardizedate_df(df_clean)

#display(df_table)

# Save  (Silver)  
df_table.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_path)          

In [0]:
from pyspark.sql.functions import col, lit
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType

# -------------------------------------------------------------
# 0) Paths
# -------------------------------------------------------------
account   = "synpasetofabric"
container = "fabmigation1"
schema_name = "Production"
table_name  = "ProductDocument"

bronze_path = f"/Volumes/main/default/fabmigration1_bronze/adventureworks/{schema_name}.{table_name}.parquet/"
base_silver_root = f"main.silver.{schema_name}_{table_name}" 

print("BRONZE:", bronze_path)
print("SILVER:", base_silver_root)

silver_path   = base_silver_root
display(silver_path)

#schema
schemaProductDocument = StructType([
    StructField("ProductID",     IntegerType(),   True),
    StructField("DocumentNode",  StringType(),    True),   # hierarchyid -> string
    StructField("ModifiedDate",  TimestampType(), True)
])

final_cols = [f.name for f in schemaProductDocument]

#read parquet
df_bronze = spark.read.parquet(bronze_path)

# -------------------------------------------------------------
# 3) Align column names
#    - If the Parquet already has the same number of columns but with
#      different/abbreviated names, rename them following schemaProduct.
#    - If columns are missing, create them as null (cast to the correct type).
# -------------------------------------------------------------
df = df_bronze
existing_cols = df.columns

# 4a.Number of columns, renamed by position
if len(existing_cols) == len(final_cols):
    for old, new in zip(existing_cols, final_cols):
        if old != new:
            df = df.withColumnRenamed(old, new)

# 4b. Create columns as null
for f in final_cols:
    if f not in df.columns:
        dtype = [sf.dataType for sf in schemaProductDocument if sf.name == f][0]
        df = df.withColumn(f, lit(None).cast(dtype))

for sf in schemaProductDocument:
    if sf.name in df.columns:
        df = df.withColumn(sf.name, col(sf.name).cast(sf.dataType))


df_final = df.select([col(c) for c in final_cols])

#clean
df_clean = clean_df(df_final)
df_table = standardizedate_df(df_clean)

#display(df_table)

# Save  (Silver)  
df_table.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_path)          

In [0]:
from pyspark.sql.functions import col, lit
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType, BinaryType

# -------------------------------------------------------------
# 0) Paths
# -------------------------------------------------------------
account   = "synpasetofabric"
container = "fabmigation1"
schema_name = "Production"
table_name  = "ProductPhoto"

bronze_path = f"/Volumes/main/default/fabmigration1_bronze/adventureworks/{schema_name}.{table_name}.parquet/"
base_silver_root = f"main.silver.{schema_name}_{table_name}" 

print("BRONZE:", bronze_path)
print("SILVER:", base_silver_root)

silver_path   = base_silver_root
display(silver_path)

#schema
schemaProductPhoto = StructType([
    StructField("ProductPhotoID",          IntegerType(),   True),
    StructField("ThumbNailPhoto",          BinaryType(),    True),   # varbinary(max) -> BinaryType
    StructField("ThumbnailPhotoFileName",  StringType(),    True),
    StructField("LargePhoto",              BinaryType(),    True),   # varbinary(max) -> BinaryType
    StructField("LargePhotoFileName",      StringType(),    True),
    StructField("ModifiedDate",            TimestampType(), True)
])

final_cols = [f.name for f in schemaProductPhoto]

#Read parquet
df_bronze = spark.read.parquet(bronze_path)

# -------------------------------------------------------------
# 3) Align column names
#    - If the Parquet already has the same number of columns but with
#      different/abbreviated names, rename them following schemaProduct.
#    - If columns are missing, create them as null (cast to the correct type).
# -------------------------------------------------------------
df = df_bronze
existing_cols = df.columns


# 4a — Number of columns, renamed by position
if len(existing_cols) == len(final_cols):
    for old, new in zip(existing_cols, final_cols):
        if old != new:
            df = df.withColumnRenamed(old, new)

# 4b — Create columns as null
for f in final_cols:
    if f not in df.columns:
        dtype = [sf.dataType for sf in schemaProductPhoto if sf.name == f][0]
        df = df.withColumn(f, lit(None).cast(dtype))


for sf in schemaProductPhoto:
    if sf.name in df.columns:
        df = df.withColumn(sf.name, col(sf.name).cast(sf.dataType))

df_final = df.select([col(c) for c in final_cols])

df_clean = clean_df(df_final)
df_table = standardizedate_df(df_clean)

#display(df_table)

# Save  (Silver)  
df_table.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_path)          

In [0]:
from pyspark.sql.functions import col, lit
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, TimestampType, BinaryType
)

# -------------------------------------------------------------
# 0) Paths
# -------------------------------------------------------------
account   = "synpasetofabric"
container = "fabmigation1"
schema_name = "Production"
table_name  = "Document"

bronze_path = f"/Volumes/main/default/fabmigration1_bronze/adventureworks/{schema_name}.{table_name}.parquet/"
base_silver_root = f"main.silver.{schema_name}_{table_name}" 

print("BRONZE:", bronze_path)
print("SILVER:", base_silver_root)

silver_path   = base_silver_root
display(silver_path)

#schema
schemaDocument = StructType([
    StructField("DocumentNode",            StringType(),    True),   # hierarchyid -> string
    # DocumentLevel  IGNORE → calculated column
    StructField("Title",                   StringType(),    True),
    StructField("Owner",                   IntegerType(),   True),
    StructField("FolderFlag",              IntegerType(),   True),   # bit -> tinyint -> integer
    StructField("FileName",                StringType(),    True),
    StructField("FileExtension",           StringType(),    True),
    StructField("Revision",                StringType(),    True),
    StructField("ChangeNumber",            IntegerType(),   True),
    StructField("Status",                  IntegerType(),   True),   # tinyint -> integer
    StructField("DocumentSummary",         StringType(),    True),   # nvarchar(max)
    StructField("Document",                BinaryType(),    True),   # varbinary(max)
    StructField("rowguid",                 StringType(),    True),   # uniqueidentifier → string
    StructField("ModifiedDate",            TimestampType(), True)
])

final_cols = [f.name for f in schemaDocument]

df_bronze = spark.read.parquet(bronze_path)

# -------------------------------------------------------------
# 3) Align column names
#    - If the Parquet already has the same number of columns but with
#      different/abbreviated names, rename them following schemaProduct.
#    - If columns are missing, create them as null (cast to the correct type).
# -------------------------------------------------------------
df = df_bronze
existing_cols = df.columns

# 4a — Number of columns, renamed by position
if len(existing_cols) == len(final_cols):
    for old, new in zip(existing_cols, final_cols):
        if old != new:
            df = df.withColumnRenamed(old, new)

# 4b — Create columns as null
for f in final_cols:
    if f not in df.columns:
        dtype = [sf.dataType for sf in schemaDocument if sf.name == f][0]
        df = df.withColumn(f, lit(None).cast(dtype))

for sf in schemaDocument:
    if sf.name in df.columns:
        df = df.withColumn(sf.name, col(sf.name).cast(sf.dataType))

df_final = df.select([col(c) for c in final_cols])

df_clean = clean_df(df_final)
df_table = standardizedate_df(df_clean)

display(df_table)

# Save  (Silver)  
df_table.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_path)          